# AOC-IDS 实验 Notebook
支持 Kaggle / Colab 两种环境。

**使用前修改下方 `GITHUB_REPO` 为你自己的仓库地址。**

In [ ]:
# ========== 配置区 ==========
GITHUB_REPO = "https://github.com/YOUR_USERNAME/AOC_IDS_NEW-main.git"  # <-- 改成你的 repo
DATASET     = "cic"     # nsl / unsw / cic
EPOCHS      = 8
FLIP        = 0.0
INTERVAL    = 2000
WINDOW      = 50        # 滑动窗口步数 (0=禁用)
THRESHOLD   = 0.3       # 置信度门控阈值 (0=禁用)
CUDA        = "0"
# ============================

In [ ]:
import os

# 检测运行环境
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in str(get_ipython().extension_manager.loaded) if hasattr(get_ipython(), 'extension_manager') else False

if ON_KAGGLE:
    WORK_DIR = '/kaggle/working'
    print('环境: Kaggle')
elif ON_COLAB:
    WORK_DIR = '/content'
    print('环境: Colab')
else:
    WORK_DIR = os.getcwd()
    print(f'环境: 本地 ({WORK_DIR})')

REPO_DIR = os.path.join(WORK_DIR, 'AOC_IDS_NEW-main')
print(f'工作目录: {REPO_DIR}')

In [ ]:
# 安装依赖 (torch 2.x + CUDA)
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-2000:])
    if result.stderr: print(result.stderr[-500:])
    return result.returncode

# torch with CUDA
run(f"{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cu121 -q")
run(f"{sys.executable} -m pip install numpy pandas scikit-learn scipy matplotlib -q")

import torch
print(f'torch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 拉取代码
import os
if not os.path.exists(REPO_DIR):
    run(f"git clone {GITHUB_REPO} {REPO_DIR}")
else:
    print('仓库已存在，执行 git pull')
    run(f"git -C {REPO_DIR} pull")

os.chdir(REPO_DIR)
print(f'当前目录: {os.getcwd()}')
print(os.listdir('.'))

In [ ]:
# ---- Kaggle 专用：从 Dataset 复制数据 ----
# 如果你把数据集上传到 Kaggle Dataset，取消注释并修改路径
# import shutil
# DATA_SRC = '/kaggle/input/your-dataset-name'
# for folder in ['CIC_pre_data', 'NSL_pre_data', 'UNSW_pre_data']:
#     src = os.path.join(DATA_SRC, folder)
#     if os.path.exists(src):
#         shutil.copytree(src, folder, dirs_exist_ok=True)
#         print(f'已复制 {folder}')

# ---- 验证数据是否存在 ----
data_dirs = {'nsl': 'NSL_pre_data', 'unsw': 'UNSW_pre_data', 'cic': 'CIC_pre_data'}
d = data_dirs[DATASET]
if os.path.exists(d):
    files = os.listdir(d)
    print(f'{d}: {files}')
else:
    print(f'[警告] 找不到数据目录 {d}，请先上传数据！')

In [ ]:
# ===== 跑 Baseline（原始代码，无优化）=====
import subprocess
cmd = (
    f"{sys.executable} -u online_training.py "
    f"--dataset {DATASET} "
    f"--epochs {EPOCHS} "
    f"--flip_percent {FLIP} "
    f"--sample_interval {INTERVAL} "
    f"--cuda {CUDA}"
)
print('命令:', cmd)
print('=' * 60)
# 实时输出
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nBaseline 退出码:', proc.returncode)

In [ ]:
# ===== 跑优化版（滑动窗口 + 置信度门控）=====
cmd = (
    f"{sys.executable} -u online_training.py "
    f"--dataset {DATASET} "
    f"--epochs {EPOCHS} "
    f"--flip_percent {FLIP} "
    f"--sample_interval {INTERVAL} "
    f"--window_steps {WINDOW} "
    f"--confidence_threshold {THRESHOLD} "
    f"--cuda {CUDA}"
)
print('命令:', cmd)
print('=' * 60)
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\n优化版 退出码:', proc.returncode)

In [ ]:
# ===== 汇总所有结果 =====
import json, glob, os
import pandas as pd

rows = []
for path in sorted(glob.glob('result/*/metrics.json')):
    with open(path) as f:
        data = json.load(f)
    cfg = data['config']
    final = data['final_results']['combined']
    rows.append({
        'run': os.path.basename(os.path.dirname(path)),
        'dataset': cfg['dataset'],
        'epochs': cfg['epochs'],
        'flip': cfg['flip_percent'],
        'window': cfg.get('window_steps', 0),
        'threshold': cfg.get('confidence_threshold', 0.0),
        'Acc': round(final['accuracy'], 4),
        'Prec': round(final['precision'], 4),
        'Recall': round(final['recall'], 4),
        'F1': round(final['f1'], 4),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ===== 打包下载结果 =====
import shutil
zip_path = os.path.join(WORK_DIR, 'aoc_ids_results')
shutil.make_archive(zip_path, 'zip', 'result')
print(f'结果已打包: {zip_path}.zip')

# Colab 下载
try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except:
    print('Kaggle: 在右侧 Output 面板下载 aoc_ids_results.zip')